In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [2]:
SAMPLES_PATH = "datasets/"

In [3]:
# Load one template per character
template_images = {}

for filename in os.listdir(SAMPLES_PATH):
    if not (filename.endswith(".png") or filename.endswith(".jpg")):
        continue

    parts = filename.split("_")
    
    if filename.startswith("cap_") and len(parts) >= 2:
        label = parts[1]  # e.g., cap_A_0.png → A
    else:
        label = parts[0]  # e.g., a_0.png → a, 5_1.png → 5

    if label not in template_images:
        img = cv2.imread(os.path.join(SAMPLES_PATH, filename), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (28, 28))
        template_images[label] = img



In [4]:
def render_text(text):
    height = 60
    width = 28 * len(text)
    canvas = 255 * np.ones((height, width), dtype=np.uint8)  # white canvas

    for i, char in enumerate(text):
        label = char
        if char.isupper():
            label = char  # Already handled as 'A', 'B' from 'cap_' files
        elif char.islower() or char.isdigit():
            label = char
        else:
            continue  # skip unsupported characters

        if label in template_images:
            img = template_images[label]
            canvas[16:44, i * 28: (i + 1) * 28] = img
    return canvas

In [5]:
def render_text_wrapped(text, chars_per_line=20):
    lines = [text[i:i+chars_per_line] for i in range(0, len(text), chars_per_line)]
    canvas = 255 * np.ones((60 * len(lines), 28 * chars_per_line), dtype=np.uint8)

    for line_idx, line in enumerate(lines):
        for i, char in enumerate(line):
            label = char.upper() if char.isupper() else char
            if label in template_images:
                img = template_images[label]
                y_offset = line_idx * 60 + 16
                x_offset = i * 28
                canvas[y_offset:y_offset+28, x_offset:x_offset+28] = img
    return canvas


In [18]:
img = render_text_wrapped("Bajrang bali ki jay kijo kesari k laal mera chota sa ye kaam ho apni raam ji se kehdena jay siyaram", chars_per_line=20)
img_upscaled = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_LINEAR)
cv2.imshow("Handwriting", img_upscaled)
cv2.waitKey(0)
cv2.destroyAllWindows()